# W06B · B · Stable Diffusion 구성요소와 CFG
2026-10-07 · 주교재5.2–5.3 · Python3.12/3.13 · 패키지0.1.10

먼저 **실제 생성 기록 분석**으로 모든 핵심 개념을 확인합니다. Colab GPU가 있으면 추가로 실제 SD 1.5를 로드하여 새 이미지를 생성합니다. 기록을 읽은 성공과 직접 생성 성공을 따로 출력합니다. 이전 노트북·API 키·개인 이미지 업로드는 필요 없습니다.

[잠재 확산](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/course/notion/w06b/w06b-latent.md) · [구성요소·CFG](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/course/notion/w06b/w06b-components.md) · [기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/course/notion/w06b/w06b-workbook.md) · [함수 정의 색인](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/W06B-API.md)


In [ ]:
import sys,subprocess,os,time
from pathlib import Path
from importlib.metadata import version,PackageNotFoundError
PUBLIC_REF='2026-fall-w06b'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.10'),('transformers','5.15.1'),('diffusers','0.40.0'),('gradio','6.26.0')])
except PackageNotFoundError:
    ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','diffusers','gradio','transformers')):
        raise RuntimeError('이전 모듈이 남아 있습니다. 런타임 재시작 후 첫 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
RUN_LIVE=IN_COLAB and torch.cuda.is_available()
print('Python',sys.version.split()[0],'package',version('luna-genai'),'CUDA',torch.cuda.is_available())
print('실제 SD 추가 생성:',RUN_LIVE,'/ 기본 핵심 활동: 실제 저장 기록 분석')
started=time.perf_counter()


## 1. 실제 기록의 출처부터 확인
[`load_recorded_sd`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L117)는 파일 해시를 검사한 기록과 이미지 목록을 반환합니다. [`load_recorded_tensors`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L130)는 실제 SD를 통과해 저장한 NumPy 배열을 읽습니다. 둘 다 SD 추론을 실행하지 않습니다. 이미지 여섯 장은 같은 공개 모델 revision으로 실제 생성했으며, 설정과 시간은 생성 당시의 기록입니다.


In [ ]:
from luna_genai.stable_diffusion import (load_recorded_sd,load_recorded_tensors,cfg_combine,load_sd15,text_condition,vae_roundtrip,generate_sd,MODEL_ID,MODEL_REVISION,DEFAULT_PROMPT)
record,gallery=load_recorded_sd();tensors=load_recorded_tensors()
display(pd.DataFrame(gallery)[['key','seed','guidance_scale','steps','device','seconds']])
print('MODEL',record['model_id'],'REVISION',record['revision'])
print('이 셀은 저장된 실측 기록을 읽었습니다. 지금 SD를 생성한 결과가 아닙니다.')


## 2. 토큰과 차원: 77은 단어 수가 아니다
아래 `input_ids`는 `(1,77)`, `hidden`은 `(1,77,768)`입니다. `is_padding`과 시작·종료 토큰을 구별합니다. 교재의 짧은 문장 `(1,7,768)`과 차원 의미를 비교하세요. 검색용 pooled embedding과도 구별합니다.


In [ ]:
token_table=pd.DataFrame(record['tokens']);display(token_table.head(22))
display(pd.DataFrame([{'array':k,'shape':tensors[k].shape} for k in ['input_ids','hidden','scaled_latent','noise_input','noise_conditional']]))
print('패딩 전 길이:',record['unpadded_length'])
plt.figure(figsize=(9,3));plt.imshow(tensors['hidden'][0,:20,:32],aspect='auto',cmap='coolwarm')
plt.xlabel('first 32 of 768 features');plt.ylabel('first 20 token positions');plt.title('Actual token-wise hidden states');plt.colorbar();plt.show()


### B1 · shape를 읽고 잘못된 설명 찾기
“77장의 이미지를 768픽셀로 표현했다”는 설명을 고치세요. 프롬프트를 더 길게 쓰면 마지막 축768이 문장 길이만큼 늘어날까요?


## 3. VAE 복원과 네 채널
기록의 `scaled_latent`는 실제 SD 1.5 VAE에 머그컵 이미지를 넣어 얻었습니다. posterior mode를 사용했습니다. `source`와 `reconstruction`은 uint8 이미지이며 기록의 MSE는 [0,1] 척도입니다. 네 채널은 RGB나 이름 붙은 독립 의미축이 아닙니다.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(8,4))
for ax,key in zip(axes,['source','reconstruction']):ax.imshow(tensors[key]);ax.set_title(key);ax.axis('off')
plt.show()
fig,axes=plt.subplots(1,4,figsize=(10,3))
for i,ax in enumerate(axes):ax.imshow(tensors['scaled_latent'][0,i],cmap='viridis');ax.set_title(f'latent channel {i}');ax.axis('off')
plt.suptitle('Each channel auto-scaled for display; colors are not RGB');plt.show()
print('VAE 기록:',record['vae'])
print('원소 수 비:',(3*512*512)/(4*64*64))


### B2 · 계산과 해석 분리
원소 수가48분의1이면 실행시간도 정확히48분의1일까요? 인코더·U-Net·디코더가 어떤 공간에서 무엇을 계산하는지 설명하세요.


### B3 · shape는 맞지만 의미가 틀린 경우
디코딩에 scaled latent를 그대로 넣는 코드를 고치려면 어떤 설정을 읽고 어떤 연산을 해야 할까요? 정답 수치0.18215를 모든 모델의 상수로 외우지는 마세요.


## 4. CFG 수계산과 실제 잡음 예측
`cfg_combine(u,c,s)`는 같은 shape의 두 유한 텐서를 `u+s*(c-u)`로 결합하며 모델을 학습하지 않습니다. 수학식 s=0과 Diffusers pipeline의 `guidance_scale<=1` 분기는 다릅니다. 실제 이미지는 scale1/4/7.5/12로 비교합니다.


In [ ]:
u=torch.tensor([.2,-.1]);c=torch.tensor([.5,.1])
display(pd.DataFrame([{'scale':s,'result':cfg_combine(u,c,s).tolist()} for s in (0.,1.,4.,7.5,12.)]))
actual_u=torch.from_numpy(tensors['noise_unconditional']);actual_c=torch.from_numpy(tensors['noise_conditional'])
actual_guided=cfg_combine(actual_u,actual_c,7.5)
print('실제 잡음 예측 shape:',actual_u.shape,'조건 변화의 평균 절대 차이:',float((actual_c-actual_u).abs().mean()))
print('이 예측은 고정 잠재잡음과 t=500의 구성요소 점검이며 이미지 품질 점수가 아닙니다.')


### B4 · 직접 계산
u=(0.1,0.4), c=(0.3,0.2), s=3일 때 차이 벡터와 최종 예측을 계산하세요. 두 벡터 사이의 평균인지도 판단하세요.


## 5. 실제 이미지 기록을 통제해서 비교
같은 조건에서 CFG만 변경한 네 장을 모두 봅니다. red→blue와 seed42→43 비교는 다음 별도 그림입니다. 충실도·결함·변화를 따로 적고 높은 CFG가 항상 우수하다고 결론 내리지 않습니다.


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,item in zip(axes,gallery[:4]):ax.imshow(Image.open(item['image_path']));ax.set_title(f"CFG {item['guidance_scale']}");ax.axis('off')
plt.show()
fig,axes=plt.subplots(1,3,figsize=(10,3))
for ax,item in zip(axes,[gallery[2],gallery[4],gallery[5]]):ax.imshow(Image.open(item['image_path']));ax.set_title(item['key']);ax.axis('off')
plt.show()


## 6. GPU 선택 경로: SD를 지금 실제로 실행
Colab 메뉴의 런타임 유형에서 T4 GPU를 선택하고 새 런타임에서 처음부터 실행하면 RUN_LIVE가 True입니다. 아래 경로는 약2.6 GB 모델을 처음 다운로드합니다. CPU 기본 경로는 기록 분석으로 끝까지 실행됩니다. 유료 업그레이드나 토큰 입력은 필요하지 않습니다.

[`load_sd15`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L30)는 revision 고정 모델과 DDIM scheduler를 준비하고 안전 검사를 유지합니다. [`text_condition`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L49)은 실제 토큰/은닉상태, [`vae_roundtrip`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L67)은 실제 encode/decode, [`generate_sd`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L87)는 실제 새 이미지를 반환합니다. 모두 모델을 재학습하지 않습니다. [`generate_sd`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/stable_diffusion.py#L87)는 실패 시 저장 이미지를 대신 반환하지 않습니다.


In [ ]:
pipe=None;live_results=[];live_condition=None;live_vae=None
if RUN_LIVE:
    pipe=load_sd15(device='cuda')
    live_condition=text_condition(pipe,DEFAULT_PROMPT)
    print('실제 GPU token/hidden:',live_condition['input_ids'].shape,live_condition['hidden'].shape)
    for scale in (1.,4.,7.5,12.):
        result=generate_sd(pipe,DEFAULT_PROMPT,seed=42,guidance=scale,steps=20)
        live_results.append(result);display(result['image']);display(result['settings'])
    live_vae=vae_roundtrip(pipe,live_results[2]['image'])
    print('실제 GPU VAE:',live_vae['scaled_latent'].shape,'MSE',live_vae['mse'])
    print('SD GPU 실제 생성 성공:',len(live_results),'장')
else:
    print('SD 직접 생성 미실행: CPU 기록 분석 모드입니다. GPU 성공으로 기록하지 않습니다.')


### B5 · 프롬프트·seed·guidance를 callback에 전달
GPU가 준비되었다면 `my_generation_callback(prompt, seed, guidance)`를 작성하여 [`live_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L55)에 세 입력과 준비된 pipe를 전달하세요. seed는 정수, guidance는 실수입니다. 아래 None을 작성한 함수로 바꾸면 마지막 버튼이 그 함수를 사용합니다. None 상태에서는 제공된 기본 callback이 작동합니다. CPU에서는 조건 전달 연습의 대상 코드를 읽고 기록 비교 버튼을 사용합니다.


In [ ]:
my_generation_callback=None  # 작성한 함수 이름으로 교체. 기본 실습 흐름은 계속 실행됩니다.


## 7. 마지막 조건부 생성 관찰실
기본3탭은 클래스 실제 생성·SD 기록 비교·CFG 수계산입니다. GPU 경로에서는 SD 직접 생성과 callback 활동 버튼을 추가합니다. [`build_conditional_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L64)은 앱 구성, [`class_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L17)는 작은 모델 생성, [`recorded_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L29)는 기록 재생, [`cfg_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L40)는 벡터 계산, [`live_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L55)는 SD 실제 생성입니다. 이름뿐 아니라 설정 출력의 mode를 확인하세요.


In [ ]:
import gradio as gr
from functools import partial
from luna_genai.conditional import load_conditional_checkpoint
from luna_genai.conditional_app import build_conditional_app,class_view,recorded_view,cfg_view,live_view,APP_CSS
class_model,class_record=load_conditional_checkpoint()
app=build_conditional_app(class_model,pipe=pipe)
if pipe is not None:
    handler=my_generation_callback or partial(live_view,pipe=pipe)
    with app:
        gr.Markdown('## B5 · 내 callback 연결\n'+('학생 함수 사용' if my_generation_callback else '제공된 기본 함수 사용 · B5 작성 후 교체 가능'))
        with gr.Row():
            p=gr.Textbox(value=DEFAULT_PROMPT,label='B5 프롬프트');s=gr.Number(value=42,precision=0,label='B5 seed');g=gr.Slider(1,12,value=4,label='B5 guidance')
        button=gr.Button('연결한 callback으로 실제 생성');out=gr.Image(label='B5 실제 생성 결과');info=gr.JSON(label='전달한 설정 확인')
        button.click(handler,[p,s,g],[out,info],api_name='student_generate')
class_image,class_settings=class_view(9,42,model=class_model)
saved_image,saved_settings=recorded_view('cfg-4')
fig,vector_settings=cfg_view(4);plt.close(fig)


In [ ]:
assert tensors['input_ids'].shape==(1,77) and tensors['hidden'].shape==(1,77,768)
assert tensors['scaled_latent'].shape==actual_guided.shape==(1,4,64,64)
assert torch.isfinite(actual_guided).all() and float((actual_c-actual_u).abs().mean())>0
assert torch.allclose(cfg_combine(u,c,1),c) and torch.equal(cfg_combine(u,c,0),u)
assert all(x['seed']==42 and x['steps']==20 for x in gallery[:4])
assert len(gallery)==6 and saved_image.size==(512,512) and '재생' in saved_settings['mode']
assert class_image.shape==(56,56) and class_settings['class_id']==9
assert np.allclose(vector_settings['guided'],[1.4,.7],atol=1e-6)
assert len(app.config['dependencies'])==(5 if RUN_LIVE else 3)
if RUN_LIVE:
    assert len(live_results)==4 and all(x['settings']['safety_checked'] for x in live_results)
    assert live_condition['hidden'].shape==(1,77,768) and live_vae['scaled_latent'].shape==(1,4,64,64)
    assert all(x['settings']['device']=='cuda' for x in live_results)
print('W06B B 검증 통과 · 기록 분석 및 callback · SD 직접 생성 여부:',RUN_LIVE)


In [ ]:
if IN_COLAB:
    app.launch(share=True,inline=False,prevent_thread_lock=True,css=APP_CSS)
else:
    print('앱 실행: app.launch(css=APP_CSS). 공개 공유 주소는 현재 런타임에 의존합니다.')


공유 주소에서 두 기록을 비교하고 실제 생성 탭이 있다면 버튼을 눌러 보세요. 다운로드·모델 준비·추론 시간을 구별합니다. 출력 mode와 설정표를 실습 기록지에 남기고, 점검 질문12개에 자기 말로 답합니다.
